Imports & Load BLIP Model

In [2]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import pandas as pd
import os
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

print("BLIP model loaded successfully!")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BLIP model loaded successfully!


 Load Dataset & Select Validation Subset

In [10]:
captions_path = "../data/captions.txt"
images_dir = "../data/Images"

df = pd.read_csv(captions_path)

VAL_SIZE = 200
val_images = df['image'].unique()[:VAL_SIZE]
print(f"Validation subset size: {len(val_images)}")

Validation subset size: 200


Function: Generate Caption

In [7]:
def generate_blip_caption(image_path):
    raw_image = Image.open(image_path).convert('RGB')
    inputs = processor(raw_image, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=30)
    caption = processor.decode(out[0], skip_special_tokens=True)
    return caption

 Run BLIP On Validation Subset

In [9]:
from nltk.tokenize import word_tokenize

results = []

for i, img_name in enumerate(val_images):
    img_path = os.path.join(images_dir, img_name)
    
    ground_truths = df[df['image'] == img_name]['caption'].tolist()
    
    generated = generate_blip_caption(img_path)
    
    results.append({
        "image": img_name,
        "generated_caption": generated,
        "ground_truth_captions": ground_truths
    })
    
    if (i+1) % 20 == 0:
        print(f"Processed {i+1}/{len(val_images)} images...")

print("Done generating captions!")
results_df = pd.DataFrame(results)

Processed 20/200 images...
Processed 40/200 images...
Processed 60/200 images...
Processed 80/200 images...
Processed 100/200 images...
Processed 120/200 images...
Processed 140/200 images...
Processed 160/200 images...
Processed 180/200 images...
Processed 200/200 images...
Done generating captions!


Compute BLEU Scores

In [12]:
smoothie = SmoothingFunction().method4

def compute_bleu(generated, references):
    gen_tokens = word_tokenize(generated.lower())
    ref_tokens = [word_tokenize(ref.lower()) for ref in references]
    return sentence_bleu(ref_tokens, gen_tokens, smoothing_function=smoothie)

results_df['bleu_score'] = results_df.apply(
    lambda row: compute_bleu(row['generated_caption'], row['ground_truth_captions']), axis=1
)

print("Average BLEU score:", results_df['bleu_score'].mean()) 

Average BLEU score: 0.19677824243736594


Compute ROUGE Scores

In [13]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(generated, references):
    # Best score across all reference captions
    best_scores = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
    for ref in references:
        scores = scorer.score(ref, generated)
        for key in best_scores:
            best_scores[key] = max(best_scores[key], scores[key].fmeasure)
    return best_scores

rouge_results = results_df.apply(
    lambda row: compute_rouge(row['generated_caption'], row['ground_truth_captions']), axis=1
)

results_df['rouge1'] = rouge_results.apply(lambda x: x['rouge1'])
results_df['rouge2'] = rouge_results.apply(lambda x: x['rouge2'])
results_df['rougeL'] = rouge_results.apply(lambda x: x['rougeL'])

print("Average ROUGE-1:", results_df['rouge1'].mean())
print("Average ROUGE-2:", results_df['rouge2'].mean())
print("Average ROUGE-L:", results_df['rougeL'].mean())

Average ROUGE-1: 0.5568843076393438
Average ROUGE-2: 0.3138433884394983
Average ROUGE-L: 0.5382722656845741


METEOR Score 

In [14]:
from nltk.translate.meteor_score import meteor_score
nltk.download('wordnet')

def compute_meteor(generated, references):
    gen_tokens = word_tokenize(generated.lower())
    ref_tokens = [word_tokenize(ref.lower()) for ref in references]
    return meteor_score(ref_tokens, gen_tokens)

results_df['meteor_score'] = results_df.apply(
    lambda row: compute_meteor(row['generated_caption'], row['ground_truth_captions']), axis=1
)

print("Average METEOR score:", results_df['meteor_score'].mean())

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...


Average METEOR score: 0.39500150220267616


Final Summary Table

In [15]:
summary_metrics = pd.DataFrame({
    "Metric": ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"],
    "Average Score": [
        round(results_df['bleu_score'].mean(), 4),
        round(results_df['rouge1'].mean(), 4),
        round(results_df['rouge2'].mean(), 4),
        round(results_df['rougeL'].mean(), 4),
        round(results_df['meteor_score'].mean(), 4) if 'meteor_score' in results_df else "N/A"
    ]
})

summary_metrics.to_csv("blip_baseline_evaluation_report.csv", index=False)
summary_metrics

,Metric,Average Score
0,BLEU,0.1968
1,ROUGE-1,0.5569
2,ROUGE-2,0.3138
3,ROUGE-L,0.5383
4,METEOR,0.3950


In [16]:
results_df.to_csv("blip_baseline_full_results.csv", index=False)
print("Saved full results and summary report")

Saved full results and summary report
